# Metadata in CyborgDB — `cyborgdb` Python SDK (service)

> ### ⚠️ Internal / pre-release
> Everything below is **unreleased**. `query_metadata`, `metadata_schema`, `pattern` fields,
> `$contains` and the datetime write path are all newer than the public 0.17.0 packages — the
> published PyPI/Docker Hub artifacts serve `/v1/vectors/query` but **404 on
> `/v1/vectors/query_metadata`** and report an empty `metadata_schema`. Do not hand this notebook
> to a customer until these ship.
>
> These builds are **not on PyPI or Docker Hub** — they ship with this notebook in the shared
> folder (see Setup). Validated against exactly these three builds:
>
> | component | version | commit |
> |---|---|---|
> | `cyborgdb-service` (Docker) | `0.17.1.dev1785938812` | `843cc3a` |
> | `cyborgdb-py` | `0.17.1.dev1785940409` | `a11eba6` |
> | `cyborgdb-core` | `0.17.1.dev1785938227` | `03db711a` |


A complete, runnable tour of **everything CyborgDB currently supports for metadata**: what you
can store, how filtering is indexed, every operator, sorting, dates, and the limits.

This notebook uses the **`cyborgdb` Python SDK** against a running **`cyborgdb-service`**
container. The SDK is a thin REST client: your app holds the index key, the service holds the
encrypted index. If you are embedding the engine in-process instead (the `cyborgdb-core` package,
no server), use the companion notebook `metadata_guide_cyborgdb_core.ipynb`; the metadata model is
identical, only the plumbing differs.

**Everything below runs in seconds** — 14 documents, 8-dimensional vectors. It is a semantics
reference, not a benchmark.

---

### What's covered

| # | Section |
|---|---|
| 1 | What can go in metadata (types) |
| 2 | `metadata_schema` — indexing policy |
| 3 | Reading metadata back |
| 4 | Two ways to filter: `query()` vs `query_metadata()` |
| 5 | Equality & set operators |
| 6 | Numeric ranges |
| 7 | `$exists` and missing-field semantics |
| 8 | Lists |
| 9 | Nested objects & dot-paths |
| 10 | Text matching: `$regex` / `$contains` |
| 11 | Logical combinators |
| 12 | Sorting: `order_by` / `ascending` / `top_k` |
| 13 | Dates |
| 14 | Limits & what is *not* supported |
| 15 | Choosing a `metadata_schema` |
| 16 | Cheat sheet |

> **Metadata is encrypted like everything else.** Values, field names, and the inverted index that
> makes filtering fast are all encrypted under your index key. The service evaluates filters
> against the encrypted index; nothing is decrypted except the rows you get back.

## 0. Setup

**1. Load and start the service.** The image ships as a tarball in the shared folder — load it
into your local Docker, then run it disk-backed. Nothing is pulled from a registry.

```bash
# pick the tarball for your machine's architecture
docker load -i docker/cyborgdb-service-0.17.1.dev1785938812-arm64.tar     # Apple Silicon / ARM
# docker load -i docker/cyborgdb-service-0.17.1.dev1785938812-amd64.tar   # Intel / AMD

docker run -it -p 8000:8000 \
  -e CYBORGDB_API_KEY=cyborg_your_api_key_here \
  -v cyborgdb_data:/app/cyborgdb_data \
  cyborgdb-service:0.17.1.dev1785938812
```

`docker load` prints the image tag it created — use that in `docker run` if it differs from the
line above.

On Linux, swap `-p 8000:8000` for `--network host`. Use `-e CYBORGDB_DB_TYPE=s3` plus the
`CYBORGDB_S3_*` variables to persist to S3 instead. `CYBORGDB_DB_TYPE=memory` is **not** usable
here: an in-memory config location can't persist the key envelope, so it rejects `create_index`.

The cells below read `CYBORGDB_BASE_URL` and `CYBORGDB_API_KEY` from the environment, so they run
unchanged against whichever container you started. The next cell checks that the service actually
exposes the metadata endpoints and stops with a clear message if it does not — which is what an
older image looks like.

**2. Install the SDK** from the wheel in the shared folder — again, **not** from PyPI, whose
`cyborgdb` 0.17.0 has no `query_metadata` and no `metadata_schema`:

```bash
pip install sdk-wheel/cyborgdb-0.17.1.dev1785940409-py3-none-any.whl
```

That one is pure Python, so the same file works on every platform. The version guard in the next
cell confirms both the SDK and the service are new enough.

Only the two CyborgDB packages come from the folder — pip still fetches their dependencies
(`numpy`, `pydantic`, `urllib3`, …) from PyPI as usual, so you need ordinary internet access, just
no registry credentials.

The `api_key` below is the service's `CYBORGDB_API_KEY` — it authenticates you *to the service*.
It is not the index key. The **index key** is the 32-byte encryption key your app owns; the
service uses it for the request and never persists it.

In [1]:
import json
import os
import re
from importlib.metadata import version as pkg_version

from cyborgdb import Client

# --- version guard -----------------------------------------------------------
# Everything in this notebook needs >= 0.17.1. The public 0.17.0 packages have no
# query_metadata / metadata_schema / $contains, so fail loudly rather than 20 cells in.
# A `0.17.1.devN` prerelease DOES satisfy this: the release tuple is what carries the
# features, so compare (major, minor, patch) and ignore the .devN suffix.
MIN_VERSION = (0, 17, 1)


def release_tuple(version: str):
    match = re.match(r"(\d+)\.(\d+)\.(\d+)", version)
    if not match:
        raise ValueError(f"unparseable version: {version!r}")
    return tuple(int(part) for part in match.groups())


def require_version(label: str, version: str):
    ok = release_tuple(version) >= MIN_VERSION
    print(f"   {'OK ' if ok else 'TOO OLD'}  {label:24s} {version}")
    if not ok:
        raise RuntimeError(
            f"{label} is {version}; this notebook needs >= "
            f"{'.'.join(map(str, MIN_VERSION))}. Install the build shipped in the shared "
            "folder (see the setup cell above) — the public release is too old."
        )

BASE_URL = os.getenv("CYBORGDB_BASE_URL", "http://localhost:8000")
API_KEY = os.getenv("CYBORGDB_API_KEY", "")

client = Client(BASE_URL, API_KEY)
health = client.get_health()

print("version check (need >= 0.17.1):")
require_version("cyborgdb (SDK)", pkg_version("cyborgdb"))
require_version("cyborgdb-service", health["version"])

# Belt and braces: an image can report a new version yet not route the endpoint.
import urllib.error
import urllib.request

try:
    spec = json.loads(urllib.request.urlopen(f"{BASE_URL}/v1/openapi.json", timeout=5).read())
    has_meta = "/v1/vectors/query_metadata" in spec["paths"]
    print(f"   {'OK ' if has_meta else 'MISSING'}  {'metadata endpoints':24s} "
          f"/v1/vectors/query_metadata")
    assert has_meta, "this service build does not route /v1/vectors/query_metadata"
except urllib.error.URLError as exc:            # non-fatal: couldn't read the spec
    print("   ??      could not read the OpenAPI spec:", exc)

# The 32-byte index key your app owns. For a real index generate a random one and
# store it in your KMS / secret manager:
#     INDEX_KEY = client.generate_key()          # == secrets.token_bytes(32)
# This notebook uses a fixed key so it can be re-run against the same index name.
INDEX_KEY = bytes(range(1, 33))
INDEX_NAME = "metadata_guide"
print("index key:", len(INDEX_KEY), "bytes")

/opt/homebrew/Caskroom/miniconda/base/envs/cyborgdb-core/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
SSL verification is disabled. Not recommended for production.


version check (need >= 0.17.1):
   OK   cyborgdb (SDK)           0.17.1.dev19
   OK   cyborgdb-service         0.17.1.dev1787320760
   OK   metadata endpoints       /v1/vectors/query_metadata
index key: 32 bytes


---
## 1. What can go in metadata

Metadata is **schemaless JSON** attached to each item at upsert time. Different items can carry
completely different fields — there is no table to declare up front, and nothing rejects an
unexpected key.

| JSON type | Indexed & filterable | Notes |
|---|---|---|
| `str` | ✅ | Equality, `$in`, and (on `pattern` fields) `$regex` / `$contains` |
| `int` / `float` | ✅ | Equality **and** numeric ranges. Both are one numeric domain — `42` matches `42.0` |
| `bool` | ✅ | Equality only (`true`/`false`), no ranges |
| `list` of scalars | ✅ | Each element is indexed under the same field → membership semantics |
| nested `dict` | ✅ | Flattened to **dot-paths**: `{"owner": {"team": "x"}}` → filter on `owner.team` |
| `datetime` | ✅ | No native date type — stored and filtered as **epoch milliseconds** (§13) |
| `null` | ❌ | Not indexed; the field reads back but behaves as **absent** to filters |
| list of objects, nested lists | ❌ | Not indexed. Restructure, or store as an opaque non-filterable field |

Two things to know about **sparse metadata**, because they drive the semantics in §7:

- A field is *present* on an item only if that item actually carried it. Missing is not `null` —
  it is absent, and filters treat it that way.
- There is no "column" to be `NULL` in. This is why `$ne` excludes items missing the field while
  `$nin` includes them.

The demo corpus below is a small document library. It is deliberately **sparse and messy** —
some items are missing `category`, `views`, `score`, or `created` — so every edge case is
observable.

In [2]:
import datetime as _dt
import re

_UTC = _dt.timezone.utc
_EPOCH = _dt.datetime(1970, 1, 1, tzinfo=_UTC)


# CyborgDB has no native date type. The REST API carries plain JSON, so convert dates to
# epoch milliseconds yourself on the way in and back on the way out. (The embedded
# `cyborgdb-core` package ships these two helpers and applies the first one automatically.)
def to_epoch_millis(value: _dt.datetime) -> int:
    if value.tzinfo is None:
        value = value.replace(tzinfo=_UTC)      # naive datetimes are treated as UTC
    return int((value - _EPOCH).total_seconds() * 1000)


def from_epoch_millis(millis: int, tz=_UTC) -> _dt.datetime:
    return _dt.datetime.fromtimestamp(millis / 1000, tz=tz)

DIM = 8

# A long title (>256 bytes) — used in §14 to show the text-matching window.
# "PAST-THE-WINDOW" starts at byte 280, so it lands outside the 256-byte match window.
LONG_TITLE = "Appendix: " + "quarterly earnings supplement " * 9 + "PAST-THE-WINDOW"

# id, and the metadata dict. Fields are intentionally missing on some items.
DOCS = [
    ("d01", {"title": "Q3 revenue forecast",             "category": "finance", "views": 1420, "score": 87.5,  "published": True,  "tags": ["revenue", "forecast"],   "owner": {"team": "finance",  "region": "emea"}, "created": to_epoch_millis(_dt.datetime(2026, 1, 15, tzinfo=_UTC)), "internal_notes": "budget owner: CFO office"}),
    ("d02", {"title": "Vector index design notes",       "category": "eng",     "views": 860,  "score": 72.0,  "published": True,  "tags": ["design", "search"],      "owner": {"team": "search",   "region": "us"},   "created": to_epoch_millis(_dt.datetime(2026, 2, 3, tzinfo=_UTC))}),
    ("d03", {"title": "Employee handbook 2026",          "category": "hr",      "views": 240,  "score": 45.25, "published": False, "tags": ["policy"],                "owner": {"team": "people",   "region": "us"},   "created": to_epoch_millis(_dt.datetime(2026, 2, 20, tzinfo=_UTC)), "internal_notes": "draft — legal review pending"}),
    ("d04", {"title": "GDPR data-processing addendum",    "category": "legal",   "views": 130,  "score": 91.0,  "published": True,  "tags": ["policy", "privacy"],     "owner": {"team": "legal",    "region": "emea"}, "created": to_epoch_millis(_dt.datetime(2026, 3, 1, tzinfo=_UTC))}),
    ("d05", {"title": "Æther migration retrospective",   "category": "eng",     "views": 410,  "score": 66.5,  "published": True,  "tags": ["design", "retro"],       "owner": {"team": "platform", "region": "apac"}, "created": to_epoch_millis(_dt.datetime(2026, 3, 18, tzinfo=_UTC))}),
    # views MISSING
    ("d06", {"title": "Pricing experiment results",      "category": "finance",                "score": 55.0,  "published": True,  "tags": ["revenue"],               "owner": {"team": "finance",  "region": "us"},   "created": to_epoch_millis(_dt.datetime(2026, 4, 2, tzinfo=_UTC))}),
    ("d07", {"title": "Encryption key rotation runbook", "category": "eng",     "views": 1980, "score": 94.0,  "published": True,  "tags": ["security", "runbook"],   "owner": {"team": "platform", "region": "emea"}, "created": to_epoch_millis(_dt.datetime(2026, 4, 25, tzinfo=_UTC)), "internal_notes": "on-call engineers only"}),
    # category, score and created MISSING
    ("d08", {"title": "Board deck notes",                                       "views": 95,                   "published": False, "tags": ["revenue"],               "owner": {"team": "finance",  "region": "us"},                                     "internal_notes": "confidential — do not distribute"}),
    ("d09", {"title": "Search relevance evaluation",     "category": "eng",     "views": 620,  "score": 78.25, "published": True,  "tags": ["search", "eval"],        "owner": {"team": "search",   "region": "apac"}, "created": to_epoch_millis(_dt.datetime(2026, 5, 11, tzinfo=_UTC))}),
    ("d10", {"title": "Vendor security questionnaire",   "category": "legal",   "views": 310,  "score": 61.0,  "published": False, "tags": ["security", "policy"],    "owner": {"team": "legal",    "region": "us"},   "created": to_epoch_millis(_dt.datetime(2026, 5, 30, tzinfo=_UTC))}),
    ("d11", {"title": "Onboarding checklist",            "category": "hr",      "views": 505,  "score": 50.0,  "published": True,  "tags": ["policy", "onboarding"],  "owner": {"team": "people",   "region": "emea"}, "created": to_epoch_millis(_dt.datetime(2026, 6, 14, tzinfo=_UTC))}),
    # almost everything MISSING — the sparse extreme
    ("d12", {"title": "Roadmap 2027 draft",                                                                    "published": False,                                                                                                                      "internal_notes": "do not share"}),
    ("d13", {"title": LONG_TITLE,                        "category": "finance", "views": 12,   "score": 30.0,  "published": False, "tags": ["revenue"],               "owner": {"team": "finance",  "region": "emea"}, "created": to_epoch_millis(_dt.datetime(2026, 6, 30, tzinfo=_UTC))}),
    # score is null -> stored, but invisible to filters (behaves as absent)
    ("d14", {"title": "Deprecated pricing sheet",         "category": "finance", "views": 7,   "score": None,  "published": False, "tags": ["revenue"],               "owner": {"team": "finance",  "region": "apac"}, "created": to_epoch_millis(_dt.datetime(2026, 7, 9, tzinfo=_UTC))}),
]

MD = dict(DOCS)                       # local copy, for printing tables below
_BASE = {"finance": 0.0, "eng": 1.0, "hr": 2.0, "legal": 3.0}


def vec_for(n, category):
    """Deterministic 8-d vector, clustered by category so filtered search is meaningful."""
    b = _BASE.get(category, 4.0)
    return [round(b + 0.01 * ((n * 7 + k * 13) % 10), 3) for k in range(DIM)]


ITEMS = [
    {"id": doc_id, "vector": vec_for(n, meta.get("category")), "metadata": meta}
    for n, (doc_id, meta) in enumerate(DOCS)
]

print(f"{len(ITEMS)} documents, {DIM}-d vectors")
print("d02 metadata:", MD["d02"])
print("d12 metadata:", MD["d12"], " <- almost every field missing")

14 documents, 8-d vectors
d02 metadata: {'title': 'Vector index design notes', 'category': 'eng', 'views': 860, 'score': 72.0, 'published': True, 'tags': ['design', 'search'], 'owner': {'team': 'search', 'region': 'us'}, 'created': 1770076800000}
d12 metadata: {'title': 'Roadmap 2027 draft', 'published': False, 'internal_notes': 'do not share'}  <- almost every field missing


---
## 2. `metadata_schema` — indexing policy, not validation

`metadata_schema` is set at index creation and is **immutable** afterwards. It does not validate
or constrain what you store — metadata stays schemaless. It only decides **how each field is
indexed**:

```python
{"field_name": {"filterable": True, "pattern": False}}
```

| Setting | Default | What it buys | What it costs |
|---|---|---|---|
| `filterable: true` | ✅ default | Inverted-index postings, so filters on the field resolve **from the index** (a pre-filter — the vector search only ever visits matching items) | Write time + index size per field |
| `filterable: false` | — | Field is still **stored** and still **filterable via `query()`** — but only through a post-filter over decrypted item blobs | Cheaper writes; much slower filtered queries; rejected by `query_metadata()` |
| `pattern: true` | ❌ off | A per-field regex dictionary, making `$regex` / `$contains` resolvable from the index | An extra dictionary per field. Requires `filterable: true` |

Fields you don't list are **filterable, non-pattern** — the index-everything default. So you only
write a `metadata_schema` to *opt out* of indexing a field, or to *opt in* to `pattern`.

For the demo index: `title` gets `pattern` (we want `$regex`/`$contains` on it), and
`internal_notes` is opted out of indexing entirely — it is free-text we never filter on, and
skipping it keeps the write path cheap.

In [3]:
METADATA_SCHEMA = {
    "title":          {"filterable": True, "pattern": True},   # + regex dictionary
    "internal_notes": {"filterable": False},                   # stored, never indexed
    # category / views / score / published / tags / owner.* / created:
    # not listed -> filterable, non-pattern (the default)
}

if INDEX_NAME in client.list_indexes():        # so the notebook can be re-run
    client.load_index(INDEX_NAME, INDEX_KEY).delete_index()

index = client.create_index(
    INDEX_NAME,
    INDEX_KEY,
    dimension=DIM,
    metric="euclidean",
    metadata_schema=METADATA_SCHEMA,
)
print("metadata_schema recorded on the index:")
for field, policy in index.metadata_schema.items():
    print(f"   {field:16s} {policy}")

metadata_schema recorded on the index:
   internal_notes   {'filterable': False, 'pattern': False, 'full_text': False}
   title            {'filterable': True, 'pattern': True, 'full_text': False}


In [4]:
index.upsert(ITEMS)
print("upserted", len(ITEMS), "items")

# A query vector near the "eng" cluster, reused throughout.
QVEC = vec_for(0, "eng")


# --- printing helpers (the CyborgDB calls below are all made directly) -------
def _path(meta, field):
    cur = meta
    for part in field.split("."):
        if not isinstance(cur, dict) or part not in cur:
            return "-"
        cur = cur[part]
    return cur


def err_text(exc):
    """One-line error message. The service returns 400s whose text sits in a JSON body."""
    msg = str(exc)
    match = re.search(r'"detail"\s*:\s*"(.*?)"\s*}', msg, re.S)
    return (match.group(1) if match else msg.splitlines()[0])[:180]


def ids_of(rows):
    """Ids out of a query_metadata() result: list[MetadataResult], i.e. {"id", ...} rows."""
    return [row["id"] for row in rows]


def show(label, rows, *fields, ordered=False):
    """Print matched ids (sorted, unless `ordered`) with selected metadata fields."""
    ids = ids_of(rows)
    ids = ids if ordered else sorted(ids)
    print(f"{label}")
    print(f"   {len(ids)} match(es): {', '.join(ids) if ids else '(none)'}")
    for doc_id in ids:
        meta = MD.get(doc_id, {})
        bits = "  ".join(f"{f}={_path(meta, f)!r}" for f in fields)
        if bits:
            print(f"      {doc_id}  {bits}")
    print()

upserted 14 items


---
## 3. Reading metadata back

Two ways to get metadata out:

- **`get(ids, include=["metadata"])`** — direct lookup by id. Returns the stored metadata dict
  verbatim, including fields that are not indexed (`internal_notes`) and `null`s.
- **`query(..., include=["metadata"])`** — attach metadata to search results. `include` also
  accepts `"distance"`; leave it out and you get ids only, which is the cheapest response.

Metadata round-trips as the JSON you wrote. The one transformation is dates → epoch millis (§13).

In [5]:
print("--- get() by id ---")
for item in index.get(["d02", "d12", "d14"], include=["metadata"]):
    print(f"   {item['id']}: {item['metadata']}")

print("\n--- query() with metadata attached ---")
for r in index.query(query_vectors=QVEC, top_k=3, include=["distance", "metadata"]):
    print(f"   {r['id']}  distance={r['distance']:.4f}  title={r['metadata']['title']!r}")

--- get() by id ---
   d02: {'title': 'Vector index design notes', 'category': 'eng', 'views': 860, 'score': 72.0, 'published': True, 'tags': ['design', 'search'], 'owner': {'team': 'search', 'region': 'us'}, 'created': 1770076800000}
   d12: {'title': 'Roadmap 2027 draft', 'published': False, 'internal_notes': 'do not share'}
   d14: {'title': 'Deprecated pricing sheet', 'category': 'finance', 'views': 7, 'score': None, 'published': False, 'tags': ['revenue'], 'owner': {'team': 'finance', 'region': 'apac'}, 'created': 1783555200000}

--- query() with metadata attached ---
   d05  distance=0.1233  title='Æther migration retrospective'
   d07  distance=0.1233  title='Encryption key rotation runbook'
   d02  distance=0.1386  title='Vector index design notes'


---
## 4. Two ways to filter

|  | `query(query_vectors=..., filters=...)` | `query_metadata(filters=...)` |
|---|---|---|
| **Question it answers** | "nearest neighbors of this vector, among items matching the filter" | "which items match the filter?" |
| **Needs a vector** | yes | no |
| **Returns** | `[{id, distance, metadata?}]`, ranked, capped by `top_k` | `[id, ...]` — **every** match unless you set `top_k` |
| **Sorting** | by vector distance | unordered, or by a metadata field via `order_by` |
| **Needs `train()`** | works untrained (exhaustive scan), faster once trained | never — the metadata index is independent of training |
| **`metadata_schema`** | **advisory.** Anything not index-resolvable falls back to a post-filter over decrypted blobs. Same rows either way, just slower | **enforced.** `$regex`/`$contains` need a `pattern` field; `filterable: false` fields are rejected outright (`ValueError`) |

The practical rule: **`query()` always answers the filter**, whatever the schema says — the
schema only decides whether it's a fast pre-filter or a slow post-filter.
**`query_metadata()` only answers what the index can resolve**, and tells you loudly when it
can't (see §14).

`query_metadata` is what you reach for to count, page, list, or sort by a metadata field — the
things a vector query can't express.

In [6]:
FILTER = {"category": "eng", "published": True}      # multiple keys AND together

print("query_metadata: the full matching set, no vector involved")
show("   {'category': 'eng', 'published': True}",
     index.query_metadata(filters=FILTER), "category", "published")

print("query: the 3 nearest of that same set")
for r in index.query(query_vectors=QVEC, top_k=3, filters=FILTER,
                     include=["distance", "metadata"]):
    print(f"      {r['id']}  distance={r['distance']:.4f}  {r['metadata']['title']!r}")

query_metadata: the full matching set, no vector involved
   {'category': 'eng', 'published': True}
   4 match(es): d02, d05, d07, d09
      d02  category='eng'  published=True
      d05  category='eng'  published=True
      d07  category='eng'  published=True
      d09  category='eng'  published=True

query: the 3 nearest of that same set
      d05  distance=0.1233  'Æther migration retrospective'
      d07  distance=0.1233  'Encryption key rotation runbook'
      d02  distance=0.1386  'Vector index design notes'


---
## 5. Equality & set operators

| Filter | Meaning |
|---|---|
| `{"category": "eng"}` | implicit `$eq` — the common case |
| `{"category": {"$eq": "eng"}}` | explicit form, identical |
| `{"category": {"$ne": "eng"}}` | has the field **and** it isn't `"eng"` |
| `{"category": {"$in": ["eng", "hr"]}}` | value is one of the listed values |
| `{"category": {"$nin": ["eng"]}}` | **not** one of them — **including items missing the field** |

Notes that matter in practice:

- **Numbers are one domain.** `{"views": 860}` matches whether it was stored as `860` or `860.0`.
- **Booleans use equality only**, never ranges: `{"published": True}`.
- **Unicode is fine** — values are matched on their exact UTF-8 bytes (`"Æther migration…"`).
- **Several operators on one field AND together**: `{"views": {"$ne": 0, "$gte": 500}}`.
- `$ne` and `$nin` differ on missing fields. That asymmetry is deliberate and covered in §7.

In [7]:
show("{'category': 'eng'}                     implicit $eq",
     index.query_metadata(filters={"category": "eng"}), "category")

show("{'category': {'$eq': 'finance'}}        explicit $eq",
     index.query_metadata(filters={"category": {"$eq": "finance"}}), "category")

show("{'views': 860}                          numeric equality",
     index.query_metadata(filters={"views": 860}), "views")

show("{'published': False}                    boolean equality",
     index.query_metadata(filters={"published": False}), "published")

show("{'title': 'Æther migration retrospective'}   unicode value",
     index.query_metadata(filters={"title": "Æther migration retrospective"}), "title")

show("{'category': {'$in': ['hr', 'legal']}}  $in",
     index.query_metadata(filters={"category": {"$in": ["hr", "legal"]}}), "category")

show("{'category': {'$ne': 'eng'}}            $ne — has the field, value != eng",
     index.query_metadata(filters={"category": {"$ne": "eng"}}), "category")

show("{'category': {'$nin': ['eng']}}         $nin — also matches items with NO category",
     index.query_metadata(filters={"category": {"$nin": ["eng"]}}), "category")

show("{'views': {'$ne': 0, '$gte': 500}}      two operators on one field -> AND",
     index.query_metadata(filters={"views": {"$ne": 0, "$gte": 500}}), "views")

{'category': 'eng'}                     implicit $eq
   4 match(es): d02, d05, d07, d09
      d02  category='eng'
      d05  category='eng'
      d07  category='eng'
      d09  category='eng'

{'category': {'$eq': 'finance'}}        explicit $eq
   4 match(es): d01, d06, d13, d14
      d01  category='finance'
      d06  category='finance'
      d13  category='finance'
      d14  category='finance'

{'views': 860}                          numeric equality
   1 match(es): d02
      d02  views=860

{'published': False}                    boolean equality
   6 match(es): d03, d08, d10, d12, d13, d14
      d03  published=False
      d08  published=False
      d10  published=False
      d12  published=False
      d13  published=False
      d14  published=False

{'title': 'Æther migration retrospective'}   unicode value
   1 match(es): d05
      d05  title='Æther migration retrospective'

{'category': {'$in': ['hr', 'legal']}}  $in
   4 match(es): d03, d04, d10, d11
      d03  category='hr'
 

---
## 6. Numeric ranges

`$gt`, `$gte`, `$lt`, `$lte` — on `int` and `float` fields (including dates, which are just
numbers; see §13).

Bounds on the **same field in the same operator object are merged into one range scan**, so
`{"views": {"$gte": 300, "$lt": 1000}}` is a single index operation, not an intersection of two.
Open-ended ranges are exactly what they look like.

Ranges are **numeric only**. `{"category": {"$gt": "eng"}}` is an error, not a lexicographic
comparison — there is no string ordering in filters (but there is in `order_by`; see §12).

In [8]:
show("{'views': {'$gte': 505}}                          inclusive — d11 (505) is in",
     index.query_metadata(filters={"views": {"$gte": 505}}), "views")

show("{'views': {'$gt': 505}}                           exclusive — d11 (505) is out",
     index.query_metadata(filters={"views": {"$gt": 505}}), "views")

show("{'views': {'$gte': 300, '$lt': 1000}}             bounded range (one scan)",
     index.query_metadata(filters={"views": {"$gte": 300, "$lt": 1000}}), "views")

show("{'score': {'$gte': 45.0, '$lte': 72.0}}           float range",
     index.query_metadata(filters={"score": {"$gte": 45.0, "$lte": 72.0}}), "score")

show("{'score': {'$lt': 50}}                            int bound against float values",
     index.query_metadata(filters={"score": {"$lt": 50}}), "score")

try:
    index.query_metadata(filters={"category": {"$gt": "eng"}})
except Exception as exc:
    print("{'category': {'$gt': 'eng'}}  -> rejected, ranges are numeric only")
    print(f"   {type(exc).__name__}: {err_text(exc)}")

{'views': {'$gte': 505}}                          inclusive — d11 (505) is in
   5 match(es): d01, d02, d07, d09, d11
      d01  views=1420
      d02  views=860
      d07  views=1980
      d09  views=620
      d11  views=505

{'views': {'$gt': 505}}                           exclusive — d11 (505) is out
   4 match(es): d01, d02, d07, d09
      d01  views=1420
      d02  views=860
      d07  views=1980
      d09  views=620

{'views': {'$gte': 300, '$lt': 1000}}             bounded range (one scan)
   5 match(es): d02, d05, d09, d10, d11
      d02  views=860
      d05  views=410
      d09  views=620
      d10  views=310
      d11  views=505

{'score': {'$gte': 45.0, '$lte': 72.0}}           float range
   6 match(es): d02, d03, d05, d06, d10, d11
      d02  score=72.0
      d03  score=45.25
      d05  score=66.5
      d06  score=55.0
      d10  score=61.0
      d11  score=50.0



Failed to query metadata: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'date': 'Fri, 21 Aug 2026 17:28:43 GMT', 'server': 'uvicorn', 'content-length': '96', 'content-type': 'application/json'})
HTTP response body: {"detail":"Failed to query metadata: Invalid input: $gt requires a numeric value, got: \"eng\""}



{'score': {'$lt': 50}}                            int bound against float values
   2 match(es): d03, d13
      d03  score=45.25
      d13  score=30.0

{'category': {'$gt': 'eng'}}  -> rejected, ranges are numeric only
   ValueError: Failed to query metadata: Invalid input: $gt requires a numeric value, got: \"eng\"


---
## 7. `$exists` and missing-field semantics

`{"field": {"$exists": True}}` matches items that carry the field; `False` matches those that
don't. Sparse metadata is the normal case, so it's worth being precise about what "missing" does
to every operator:

| Filter on `category` | Item **missing** `category` |
|---|---|
| `{"category": "eng"}` (implicit `$eq`) | ❌ not a match |
| `{"$eq": ...}` | ❌ not a match |
| `{"$ne": "eng"}` | ❌ **not** a match — `$ne` requires the field to be present |
| `{"$in": [...]}` | ❌ not a match |
| `{"$nin": [...]}` | ✅ **is** a match — `$nin` is a complement |
| `{"$gt"/"$gte"/"$lt"/"$lte": ...}` | ❌ not a match |
| `{"$regex"/"$contains": ...}` | ❌ not a match |
| `{"$exists": True}` | ❌ not a match |
| `{"$exists": False}` | ✅ is a match |
| `{"$not": {"category": "eng"}}` | ✅ is a match — complement of the inner filter |
| `{"$nor": [{"category": "eng"}]}` | ✅ is a match — also a complement |

**Rule of thumb:** positive operators require the field. Complements (`$nin`, `$exists: false`,
`$not`, `$nor`) include items that don't have it. When you want "has the field *and* isn't X",
scope the negation explicitly:

```python
{"$and": [{"category": {"$exists": True}}, {"category": {"$nin": ["eng"]}}]}
```

**`null` counts as missing.** `d14` stores `"score": None`. It reads back as `null` from `get()`,
but it is not indexed, so `{"score": {"$exists": True}}` does not match it.

In [9]:
show("{'category': {'$exists': True}}",
     index.query_metadata(filters={"category": {"$exists": True}}), "category")
show("{'category': {'$exists': False}}",
     index.query_metadata(filters={"category": {"$exists": False}}), "title")
show("{'score': {'$exists': True}}   <- d14 stores score=None, so it is NOT here",
     index.query_metadata(filters={"score": {"$exists": True}}), "score")

# The table above, verified against the index.
NO_CATEGORY = {"d08", "d12"}
CASES = [
    ("{'category': 'eng'}",                     {"category": "eng"}),
    ("{'category': {'$eq': 'eng'}}",            {"category": {"$eq": "eng"}}),
    ("{'category': {'$ne': 'eng'}}",            {"category": {"$ne": "eng"}}),
    ("{'category': {'$in': ['eng']}}",          {"category": {"$in": ["eng"]}}),
    ("{'category': {'$nin': ['eng']}}",         {"category": {"$nin": ["eng"]}}),
    ("{'category': {'$exists': True}}",         {"category": {"$exists": True}}),
    ("{'category': {'$exists': False}}",        {"category": {"$exists": False}}),
    ("{'$not': {'category': 'eng'}}",           {"$not": {"category": "eng"}}),
    ("{'$nor': [{'category': 'eng'}]}",         {"$nor": [{"category": "eng"}]}),
]
print(f"{'filter':44s} {'matches':>7s}   items missing `category` that matched")
print("-" * 96)
for label, filt in CASES:
    ids = set(ids_of(index.query_metadata(filters=filt)))
    hit = sorted(ids & NO_CATEGORY)
    print(f"{label:44s} {len(ids):>7d}   {', '.join(hit) if hit else '-'}")

print("\nscoped negation — has the field AND is not 'eng':")
show("   {'$and': [{'category': {'$exists': True}}, {'category': {'$nin': ['eng']}}]}",
     index.query_metadata(filters={"$and": [{"category": {"$exists": True}}, {"category": {"$nin": ["eng"]}}]}),
     "category")

{'category': {'$exists': True}}
   12 match(es): d01, d02, d03, d04, d05, d06, d07, d09, d10, d11, d13, d14
      d01  category='finance'
      d02  category='eng'
      d03  category='hr'
      d04  category='legal'
      d05  category='eng'
      d06  category='finance'
      d07  category='eng'
      d09  category='eng'
      d10  category='legal'
      d11  category='hr'
      d13  category='finance'
      d14  category='finance'

{'category': {'$exists': False}}
   2 match(es): d08, d12
      d08  title='Board deck notes'
      d12  title='Roadmap 2027 draft'

{'score': {'$exists': True}}   <- d14 stores score=None, so it is NOT here
   11 match(es): d01, d02, d03, d04, d05, d06, d07, d09, d10, d11, d13
      d01  score=87.5
      d02  score=72.0
      d03  score=45.25
      d04  score=91.0
      d05  score=66.5
      d06  score=55.0
      d07  score=94.0
      d09  score=78.25
      d10  score=61.0
      d11  score=50.0
      d13  score=30.0

filter                               

---
## 8. Lists

A list value indexes **each element** under the same field, which gives you membership
semantics for free — there is no separate "array" operator:

| Filter | Meaning |
|---|---|
| `{"tags": "policy"}` | the list **contains** `"policy"` |
| `{"tags": {"$in": ["search", "privacy"]}}` | contains **any** of these |
| `{"tags": {"$nin": ["policy"]}}` | contains none of these (or has no `tags` at all) |

Lists of numbers work the same way, and a range matches if **any** element falls inside it.

There is no built-in "contains all" — express it as an `$and` of single-value matches:
`{"$and": [{"tags": "design"}, {"tags": "search"}]}`.

In [10]:
show("{'tags': 'policy'}                          list membership",
     index.query_metadata(filters={"tags": "policy"}), "tags")

show("{'tags': {'$in': ['search', 'privacy']}}    any-of",
     index.query_metadata(filters={"tags": {"$in": ["search", "privacy"]}}), "tags")

show("{'tags': {'$nin': ['policy', 'revenue']}}   none-of (+ items with no tags)",
     index.query_metadata(filters={"tags": {"$nin": ["policy", "revenue"]}}), "tags")

show("$and of two memberships                     contains BOTH design and search",
     index.query_metadata(filters={"$and": [{"tags": "design"}, {"tags": "search"}]}), "tags")

{'tags': 'policy'}                          list membership
   4 match(es): d03, d04, d10, d11
      d03  tags=['policy']
      d04  tags=['policy', 'privacy']
      d10  tags=['security', 'policy']
      d11  tags=['policy', 'onboarding']

{'tags': {'$in': ['search', 'privacy']}}    any-of
   3 match(es): d02, d04, d09
      d02  tags=['design', 'search']
      d04  tags=['policy', 'privacy']
      d09  tags=['search', 'eval']

{'tags': {'$nin': ['policy', 'revenue']}}   none-of (+ items with no tags)
   5 match(es): d02, d05, d07, d09, d12
      d02  tags=['design', 'search']
      d05  tags=['design', 'retro']
      d07  tags=['security', 'runbook']
      d09  tags=['search', 'eval']
      d12  tags='-'

$and of two memberships                     contains BOTH design and search
   1 match(es): d02
      d02  tags=['design', 'search']



---
## 9. Nested objects & dot-paths

Nested dicts are flattened to dot-paths at write time, so `{"owner": {"team": "search"}}` is
filterable as `owner.team`. Nesting can go arbitrarily deep (`a.b.c.d`), and every operator works
on a dot-path exactly as it does on a top-level field — including `$exists` and ranges.

Two things to keep in mind:

- **Filter on the leaf, not the object.** `{"owner": {"team": "search"}}` is not a nested-equality
  filter — a dict value where an operator object is expected is read as an operator object, and
  `team` isn't an operator, so it errors. Write `{"owner.team": "search"}`.
- **Lists of objects are not indexed.** `{"authors": [{"name": "a"}, {"name": "b"}]}` gives you no
  `authors.name` field. Restructure to parallel scalar lists (`{"author_names": ["a", "b"]}`) if
  you need to filter on it.

In [11]:
show("{'owner.team': 'search'}                       nested equality",
     index.query_metadata(filters={"owner.team": "search"}), "owner.team", "owner.region")

show("{'owner.region': {'$in': ['emea', 'apac']}}    nested $in",
     index.query_metadata(filters={"owner.region": {"$in": ["emea", "apac"]}}), "owner.region")

show("{'owner.region': {'$exists': False}}           no owner at all",
     index.query_metadata(filters={"owner.region": {"$exists": False}}), "title")

show("dot-path + range combined",
     index.query_metadata(filters={"$and": [{"owner.region": "emea"}, {"views": {"$gte": 400}}]}),
     "owner.region", "views")

try:
    index.query_metadata(filters={"owner": {"team": "search"}})
except Exception as exc:
    print("{'owner': {'team': 'search'}}  -> rejected: filter the leaf path, not the object")
    print(f"   {type(exc).__name__}: {err_text(exc)}")

Failed to query metadata: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'date': 'Fri, 21 Aug 2026 17:28:43 GMT', 'server': 'uvicorn', 'content-length': '89', 'content-type': 'application/json'})
HTTP response body: {"detail":"Failed to query metadata: Invalid input: Unsupported metadata operator: team"}



{'owner.team': 'search'}                       nested equality
   2 match(es): d02, d09
      d02  owner.team='search'  owner.region='us'
      d09  owner.team='search'  owner.region='apac'

{'owner.region': {'$in': ['emea', 'apac']}}    nested $in
   8 match(es): d01, d04, d05, d07, d09, d11, d13, d14
      d01  owner.region='emea'
      d04  owner.region='emea'
      d05  owner.region='apac'
      d07  owner.region='emea'
      d09  owner.region='apac'
      d11  owner.region='emea'
      d13  owner.region='emea'
      d14  owner.region='apac'

{'owner.region': {'$exists': False}}           no owner at all
   1 match(es): d12
      d12  title='Roadmap 2027 draft'

dot-path + range combined
   3 match(es): d01, d07, d11
      d01  owner.region='emea'  views=1420
      d07  owner.region='emea'  views=1980
      d11  owner.region='emea'  views=505

{'owner': {'team': 'search'}}  -> rejected: filter the leaf path, not the object
   ValueError: Failed to query metadata: Invalid input: Uns

---
## 10. Text matching — `$regex` and `$contains`

| Filter | Meaning |
|---|---|
| `{"title": {"$regex": "^Vector"}}` | ECMAScript regex, **unanchored search** — it matches anywhere unless you anchor it |
| `{"title": {"$contains": "security"}}` | plain, case-sensitive substring. Cheaper than a regex and needs no escaping |

Both are **only index-resolvable on a `pattern: true` field**, which is why `title` is declared
that way in §2. This is the one place where `metadata_schema` changes behavior rather than just
speed:

- **`query()`** works on any field. On a `pattern` field it resolves from the regex dictionary;
  otherwise it falls back to a post-filter over decrypted metadata — correct, but it pays for
  decrypting candidates.
- **`query_metadata()`** requires `pattern: true` and raises `ValueError` otherwise.

Two limits worth internalizing:

- **Matching sees the first 256 bytes of a value only.** Longer values are truncated in the
  pattern dictionary, and the post-filter clamps to the same window so both paths agree. Text
  past that boundary is invisible to `$regex`/`$contains` (§14).
- **On a list field, it matches if any element matches.**

In [12]:
show("{'title': {'$regex': '^Vector'}}              anchored prefix",
     index.query_metadata(filters={"title": {"$regex": "^Vector"}}), "title")

show("{'title': {'$regex': 'runbook|checklist'}}    alternation, unanchored",
     index.query_metadata(filters={"title": {"$regex": "runbook|checklist"}}), "title")

show("{'title': {'$regex': '20[0-9]{2}$'}}          character class + anchor",
     index.query_metadata(filters={"title": {"$regex": "20[0-9]{2}$"}}), "title")

show("{'title': {'$contains': 'security'}}          substring, case-sensitive",
     index.query_metadata(filters={"title": {"$contains": "security"}}), "title")

show("{'title': {'$contains': 'Security'}}          ... so casing matters",
     index.query_metadata(filters={"title": {"$contains": "Security"}}), "title")

# On a non-pattern field, query() still answers -- via the post-filter path.
print("$regex on `category` (not a pattern field):")
print("   query_metadata ->", end=" ")
try:
    index.query_metadata(filters={"category": {"$regex": "^eng"}})
    print("resolved")
except Exception as exc:
    print(f"{type(exc).__name__} (needs pattern: true)")
res = index.query(query_vectors=QVEC, top_k=10, filters={"category": {"$regex": "^eng"}},
                  include=["distance", "metadata"])
print("   query        ->", len(res), "results:", ", ".join(sorted(r["id"] for r in res)))

Failed to query metadata: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'date': 'Fri, 21 Aug 2026 17:28:43 GMT', 'server': 'uvicorn', 'content-length': '115', 'content-type': 'application/json'})
HTTP response body: {"detail":"Failed to query metadata: Invalid input: $regex on 'category' requires a regex-indexed (pattern) field"}



{'title': {'$regex': '^Vector'}}              anchored prefix
   1 match(es): d02
      d02  title='Vector index design notes'

{'title': {'$regex': 'runbook|checklist'}}    alternation, unanchored
   2 match(es): d07, d11
      d07  title='Encryption key rotation runbook'
      d11  title='Onboarding checklist'

{'title': {'$regex': '20[0-9]{2}$'}}          character class + anchor
   1 match(es): d03
      d03  title='Employee handbook 2026'

{'title': {'$contains': 'security'}}          substring, case-sensitive
   1 match(es): d10
      d10  title='Vendor security questionnaire'

{'title': {'$contains': 'Security'}}          ... so casing matters
   0 match(es): (none)

$regex on `category` (not a pattern field):
   query_metadata -> ValueError (needs pattern: true)
   query        -> 4 results: d02, d05, d07, d09


---
## 11. Logical combinators

| Operator | Meaning |
|---|---|
| *(multiple top-level keys)* | implicit `$and` |
| `{"$and": [f1, f2, ...]}` | all must match |
| `{"$or": [f1, f2, ...]}` | at least one matches |
| `{"$not": {...}}` | complement of one filter object |
| `{"$nor": [f1, f2, ...]}` | none match — complement of `$or` |

They nest freely: any branch can be another combinator, and combinators can hold field
conditions, dot-paths, ranges, regexes — anything from the sections above.

Edge cases, so nothing surprises you: `{"$and": []}` matches **everything** (vacuous truth),
`{"$or": []}` matches **nothing**, and `{}` matches everything. `$not` and `$nor` are
complements, so they include items missing the referenced fields (§7).

In [13]:
show("implicit AND: {'category': 'eng', 'published': True}",
     index.query_metadata(filters={"category": "eng", "published": True}),
     "category", "published")

show("$and: eng AND views >= 600",
     index.query_metadata(filters={"$and": [{"category": "eng"}, {"views": {"$gte": 600}}]}),
     "category", "views")

show("$or: legal OR hr",
     index.query_metadata(filters={"$or": [{"category": "legal"}, {"category": "hr"}]}),
     "category")

show("$not: NOT (views >= 500) — a complement, so d06/d12 (no `views`) are IN",
     index.query_metadata(filters={"$not": {"views": {"$gte": 500}}}), "views")

show("$nor: neither eng nor finance",
     index.query_metadata(filters={"$nor": [{"category": "eng"}, {"category": "finance"}]}),
     "category")

show("nested: emea AND (high views OR high score)",
     index.query_metadata(filters={"$and": [
         {"owner.region": "emea"},
         {"$or": [{"views": {"$gte": 1000}}, {"score": {"$gte": 90}}]},
     ]}), "owner.region", "views", "score")

show("nested deeper: published eng/finance, tagged, not owned by `search`",
     index.query_metadata(filters={"$and": [
         {"published": True},
         {"category": {"$in": ["eng", "finance"]}},
         {"tags": {"$exists": True}},
         {"$not": {"owner.team": "search"}},
     ]}), "category", "owner.team", "tags")

print("edge cases:")
print(f"   {'{}':12s} -> {len(index.query_metadata(filters={})):2d} matches (everything)")
print(f"   {"{'$and': []}":12s} -> {len(index.query_metadata(filters={'$and': []})):2d} matches (AND of nothing is true)")
print(f"   {"{'$or': []}":12s} -> {len(index.query_metadata(filters={'$or': []})):2d} matches (empty union)")

implicit AND: {'category': 'eng', 'published': True}
   4 match(es): d02, d05, d07, d09
      d02  category='eng'  published=True
      d05  category='eng'  published=True
      d07  category='eng'  published=True
      d09  category='eng'  published=True

$and: eng AND views >= 600
   3 match(es): d02, d07, d09
      d02  category='eng'  views=860
      d07  category='eng'  views=1980
      d09  category='eng'  views=620

$or: legal OR hr
   4 match(es): d03, d04, d10, d11
      d03  category='hr'
      d04  category='legal'
      d10  category='legal'
      d11  category='hr'

$not: NOT (views >= 500) — a complement, so d06/d12 (no `views`) are IN
   9 match(es): d03, d04, d05, d06, d08, d10, d12, d13, d14
      d03  views=240
      d04  views=130
      d05  views=410
      d06  views='-'
      d08  views=95
      d10  views=310
      d12  views='-'
      d13  views=12
      d14  views=7

$nor: neither eng nor finance
   6 match(es): d03, d04, d08, d10, d11, d12
      d03  category='

---
## 12. Sorting — `order_by`, `ascending`, `top_k`

`query_metadata` can sort its matches by a metadata field. (A vector `query()` is always ranked
by distance — that's its job — so sorting lives here.)

```python
index.query_metadata(filters={...}, order_by="views", ascending=False, top_k=5)
```

- `order_by` takes a field name plus `ascending`, or a MongoDB-style single-field dict:
  `order_by={"views": -1}` is the same as `order_by="views", ascending=False`.
- Sorting is applied **post-filter**: filter first, then sort the matches.
- **`top_k` is applied *after* the sort**, so `order_by` + `top_k` gives you a genuine top-N.
  Without `order_by` the result is an unordered subset and `top_k` truncates it arbitrarily.
- **Mixed types have a defined order**: `number < string < bool`.
- **Items missing the field — or holding a non-scalar (list/dict/null) — sort last in *both*
  directions.** `ascending` flips only the ordering of present, scalar values. This is what you
  want for "top 10 by views": items with no `views` never lead the list.

Sorting decrypts the matched items' metadata to read the sort key, so it costs
*O(matches)*, not *O(index)*. Filter first, sort second.

In [14]:
show("order_by='views', descending — note d06/d12 (no views) sort LAST",
     index.query_metadata(filters={}, order_by="views", ascending=False), "views", ordered=True)

show("order_by='views', ascending — missing values STILL last",
     index.query_metadata(filters={}, order_by="views", ascending=True), "views", ordered=True)

show("top 3 by score (top_k applied AFTER the sort)",
     index.query_metadata(filters={}, order_by="score", ascending=False, top_k=3), "score", ordered=True)

show("order_by={'views': -1} — MongoDB-style shorthand",
     index.query_metadata(filters={}, order_by={"views": -1}, top_k=3), "views", ordered=True)

show("filter, then sort: published eng docs by score",
     index.query_metadata(filters={"category": "eng", "published": True}, order_by="score", ascending=False),
     "score", ordered=True)

show("string field ascending",
     index.query_metadata(filters={"category": {"$exists": True}}, order_by="category"), "category", ordered=True)

show("bool field ascending — False before True",
     index.query_metadata(filters={}, order_by="published"), "published", ordered=True)

show("non-scalar sort key: `tags` is a list -> unorderable, so top_k truncates arbitrarily",
     index.query_metadata(filters={}, order_by="tags", top_k=5), "tags", ordered=True)

order_by='views', descending — note d06/d12 (no views) sort LAST
   14 match(es): d07, d01, d02, d09, d11, d05, d10, d03, d04, d08, d13, d14, d12, d06
      d07  views=1980
      d01  views=1420
      d02  views=860
      d09  views=620
      d11  views=505
      d05  views=410
      d10  views=310
      d03  views=240
      d04  views=130
      d08  views=95
      d13  views=12
      d14  views=7
      d12  views='-'
      d06  views='-'

order_by='views', ascending — missing values STILL last
   14 match(es): d14, d13, d08, d04, d03, d10, d05, d11, d09, d02, d01, d07, d12, d06
      d14  views=7
      d13  views=12
      d08  views=95
      d04  views=130
      d03  views=240
      d10  views=310
      d05  views=410
      d11  views=505
      d09  views=620
      d02  views=860
      d01  views=1420
      d07  views=1980
      d12  views='-'
      d06  views='-'

top 3 by score (top_k applied AFTER the sort)
   3 match(es): d07, d04, d01
      d07  score=94.0
      d04  score=91.0
 

---
## 13. Dates

**CyborgDB has no native date type.** Dates are stored as **epoch milliseconds** (an integer), so
everything numeric applies: `$gt/$gte/$lt/$lte` ranges, `$eq`, `$in`, and `order_by`.

The REST API carries plain JSON, so the SDK does **not** convert `datetime` objects for you —
convert them yourself at the boundary, both when writing and inside filters. The `to_epoch_millis`
/ `from_epoch_millis` helpers defined in §1 are all you need; the embedded `cyborgdb-core` package
ships the same two and applies the write-side one automatically.

Pin down one convention and stick to it (UTC is the sane default), because an epoch integer
carries no timezone. Mixing local-time and UTC conversions silently shifts your range boundaries.

In [15]:
# The corpus wrote to_epoch_millis(...) values; they come back as integers.
stored = index.get(["d04"], include=["metadata"])[0]["metadata"]["created"]
print("stored value:", stored, f"({type(stored).__name__})")
print("back to datetime:", from_epoch_millis(stored).isoformat())

# Convert filter bounds the same way you converted the stored values.
q2 = to_epoch_millis(_dt.datetime(2026, 4, 1, tzinfo=_UTC))
q3 = to_epoch_millis(_dt.datetime(2026, 7, 1, tzinfo=_UTC))

show("created in [2026-04-01, 2026-07-01)  — epoch-millis bounds",
     index.query_metadata(filters={"created": {"$gte": q2, "$lt": q3}}), "created")

show("created before 2026-03-01",
     index.query_metadata(filters={"created": {"$lt": to_epoch_millis(_dt.datetime(2026, 3, 1, tzinfo=_UTC))}}),
     "created")

show("no `created` at all",
     index.query_metadata(filters={"created": {"$exists": False}}), "title")

print("newest first (order_by on a date field; undated items sort last):")
for doc_id in ids_of(index.query_metadata(filters={}, order_by="created", ascending=False)):
    raw = MD[doc_id].get("created")
    stamp = from_epoch_millis(raw).date().isoformat() if raw is not None else "        - "
    print(f"   {doc_id}  {stamp}  {MD[doc_id]['title'][:44]!r}")

stored value: 1772323200000 (int)
back to datetime: 2026-03-01T00:00:00+00:00
created in [2026-04-01, 2026-07-01)  — epoch-millis bounds
   6 match(es): d06, d07, d09, d10, d11, d13
      d06  created=1775088000000
      d07  created=1777075200000
      d09  created=1778457600000
      d10  created=1780099200000
      d11  created=1781395200000
      d13  created=1782777600000

created before 2026-03-01
   3 match(es): d01, d02, d03
      d01  created=1768435200000
      d02  created=1770076800000
      d03  created=1771545600000

no `created` at all
   2 match(es): d08, d12
      d08  title='Board deck notes'
      d12  title='Roadmap 2027 draft'

newest first (order_by on a date field; undated items sort last):
   d14  2026-07-09  'Deprecated pricing sheet'
   d13  2026-06-30  'Appendix: quarterly earnings supplement quar'
   d11  2026-06-14  'Onboarding checklist'
   d10  2026-05-30  'Vendor security questionnaire'
   d09  2026-05-11  'Search relevance evaluation'
   d07  2026-04-25

---
## 14. Limits & what is *not* supported

**Operators that don't exist.** `$type` is explicitly rejected; so is any unknown `$operator` —
you get an error rather than a silently-ignored clause. There is no `$size`, `$elemMatch`,
`$all`, `$mod`, or `$where`.

**`query_metadata` is strict; `query` is not.** `query_metadata` must resolve the whole filter
from the metadata index, so the service returns 400 and the SDK raises `ValueError` when it can't:

| Situation | `query_metadata` | `query` (with a vector) |
|---|---|---|
| `$regex`/`$contains` on a non-`pattern` field | ❌ raises | ✅ post-filter |
| any filter on a `filterable: false` field | ❌ raises | ✅ post-filter |
| `$regex`/`$contains` matching a value stored **truncated** (>256 B) | ❌ raises | ✅ post-filter |

**Text matching sees the first 256 bytes of a value.** On a `pattern` field, that prefix *is* the
window `$regex`/`$contains` are defined over, and the post-filter clamps to the same window so
both paths agree. Two consequences: text past byte 256 is a genuine non-match on either path, and
a match *inside* the window on a value that was truncated can't be mapped back to ids from the
dictionary alone — `query_metadata` raises, and you use `query()` for it.

**`metadata_schema` is immutable.** It's fixed at `create_index` and can't be altered later.
Changing indexing policy means creating a new index and re-upserting.

**Not indexed** (stored and returned, but invisible to filters): `null` values, lists of objects,
nested lists.

**Sizing.** Metadata is stored per item and the inverted index grows with the number of distinct
values per field, so a unique-per-item field is a large index. Use `filterable: false` for large
free-text you never filter on.

In [16]:
def expect_error(label, fn):
    try:
        result = fn()
    except Exception as exc:
        print(f"   ✅ rejected  {label}")
        print(f"                {type(exc).__name__}: {err_text(exc)}")
    else:
        print(f"   ⚠️  accepted  {label} -> {result}")


print("query_metadata rejects what it cannot resolve from the index:")
expect_error("$type (not supported at all)",
             lambda: index.query_metadata(filters={"views": {"$type": "number"}}))
expect_error("$size (no such operator)",
             lambda: index.query_metadata(filters={"tags": {"$size": 2}}))
expect_error("$regex on `category` (not a pattern field)",
             lambda: index.query_metadata(filters={"category": {"$regex": "^eng"}}))
expect_error("any filter on `internal_notes` (filterable: False)",
             lambda: index.query_metadata(filters={"internal_notes": {"$exists": True}}))
expect_error("$regex matching a >256 B value (d13's long title)",
             lambda: index.query_metadata(filters={"title": {"$regex": "^Appendix"}}))

print("\nquery() answers all of those via the post-filter path:")
for label, filt in [
    ("$regex on non-pattern `category`", {"category": {"$regex": "^eng"}}),
    ("filterable:False `internal_notes`", {"internal_notes": {"$contains": "confidential"}}),
    ("$regex on the >256 B title",        {"title": {"$regex": "^Appendix"}}),
]:
    results = index.query(query_vectors=QVEC, top_k=14, filters=filt,
                          include=["distance", "metadata"])
    ids = sorted(r["id"] for r in results)
    print(f"   {label:36s} -> {len(ids)} result(s): {', '.join(ids) or '(none)'}")

print("\ntext past byte 256 is invisible on BOTH paths (a real non-match, not an error):")
marker = "PAST-THE-WINDOW"        # starts at byte 280 of d13's title
print(f"   d13 title is {len(MD['d13']['title'])} bytes; {marker!r} starts at byte "
      f"{MD['d13']['title'].index(marker)} — outside the window")

marker_filter = {"title": {"$contains": marker}}
meta_hits = index.query_metadata(filters=marker_filter)
vector_hits = index.query(query_vectors=QVEC, top_k=14, filters=marker_filter,
                          include=["distance", "metadata"])
print(f"   query_metadata -> {len(meta_hits)} match(es)   <- not an error, a genuine non-match")
print(f"   query          -> {len(vector_hits)} result(s)")

print("\n`null` is stored but not indexed:")
stored_score = index.get(["d14"], include=["metadata"])[0]["metadata"]["score"]
has_score = ids_of(index.query_metadata(filters={"score": {"$exists": True}}))
print("   d14 metadata from get():", stored_score)
print("   {'score': {'$exists': True}} includes d14?", "d14" in has_score)

query_metadata rejects what it cannot resolve from the index:


Failed to query metadata: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'date': 'Fri, 21 Aug 2026 17:28:44 GMT', 'server': 'uvicorn', 'content-length': '76', 'content-type': 'application/json'})
HTTP response body: {"detail":"Failed to query metadata: Invalid input: $type is not supported"}

Failed to query metadata: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'date': 'Fri, 21 Aug 2026 17:28:44 GMT', 'server': 'uvicorn', 'content-length': '90', 'content-type': 'application/json'})
HTTP response body: {"detail":"Failed to query metadata: Invalid input: Unsupported metadata operator: $size"}

Failed to query metadata: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'date': 'Fri, 21 Aug 2026 17:28:44 GMT', 'server': 'uvicorn', 'content-length': '115', 'content-type': 'application/json'})
HTTP response body: {"detail":"Failed to query metadata: Invalid input: $regex on 'category' requires a regex-indexed (pattern) field"}

Failed to 

   ✅ rejected  $type (not supported at all)
                ValueError: Failed to query metadata: Invalid input: $type is not supported
   ✅ rejected  $size (no such operator)
                ValueError: Failed to query metadata: Invalid input: Unsupported metadata operator: $size
   ✅ rejected  $regex on `category` (not a pattern field)
                ValueError: Failed to query metadata: Invalid input: $regex on 'category' requires a regex-indexed (pattern) field
   ✅ rejected  any filter on `internal_notes` (filterable: False)
                ValueError: Failed to query metadata: Invalid input: query_metadata cannot filter on a non-indexed metadata field; index the field or use query() with a vector
   ✅ rejected  $regex matching a >256 B value (d13's long title)
                ValueError: Failed to query metadata: Invalid input: $regex matched a value stored truncated (>256B); not resolvable in query_metadata

query() answers all of those via the post-filter path:
   $regex on no

---
## 15. Choosing a `metadata_schema`

The default (index everything, no pattern dictionaries) is right for most indexes. Reach for an
explicit schema in these cases:

| You have | Do this | Why |
|---|---|---|
| A field you filter on constantly (`tenant_id`, `category`, `status`) | leave it default (`filterable: true`) | Pre-filter: the vector search only visits matching items |
| Large free text you never filter on (chunk bodies, raw payloads, notes) | `{"filterable": false}` | Skips postings entirely — cheaper writes and a smaller index. Still stored, still returned, still `query()`-filterable if you ever need it |
| A field you need `$regex` / `$contains` on | `{"filterable": true, "pattern": true}` | Without it, text matching falls back to a post-filter and `query_metadata` refuses |
| A unique-per-item value you only ever look up by id (a hash, a URL) | `{"filterable": false}` | A unique-per-item field means a posting list per item — all cost, no selectivity |
| A high-cardinality field you *do* filter on (`user_id`) | leave it default | Equality on a high-cardinality field is very selective, which is exactly what a pre-filter is good at |

Beyond the schema:

- **Selectivity is what makes filtered search fast.** A filter matching 0.1% of the index lets the
  vector search skip almost everything; one matching 90% saves nothing. Put the selective clause
  in the filter and the rest in post-processing if you must.
- **Prefer flat scalars and dot-paths** over deep structures. Both are indexed identically, but
  flat is easier to reason about — and lists of objects aren't indexed at all.
- **Prefer `$contains` over `$regex`** for plain substring matching. Same result, less work, no
  escaping bugs.
- **Filter first, sort second.** `order_by` decrypts the matched set, so it scales with matches,
  not index size.
- **Store dates as epoch millis, in UTC, consistently.**

---
## 16. Cheat sheet

```python
# --- create: metadata_schema is indexing policy, and immutable ---------------
index = client.create_index(
    "docs", INDEX_KEY, dimension=768,
    metadata_schema={
        "title": {"filterable": True, "pattern": True},   # enables $regex / $contains
        "body":  {"filterable": False},                   # stored, not indexed
    },
)
index.metadata_schema                       # read it back

# --- write: schemaless JSON per item ----------------------------------------
index.upsert([{"id": "d1", "vector": vec, "metadata": {
    "title": "Q3 forecast",              # str
    "views": 1420,                       # int / float -> ranges
    "published": True,                   # bool
    "tags": ["revenue", "forecast"],     # list -> membership
    "owner": {"team": "finance"},        # nested -> filter as "owner.team"
    "created": 1767225600000,            # dates = epoch millis
}}])

# --- filter: vector search, ranked by distance ------------------------------
index.query(query_vectors=qv, top_k=10,
            filters={"category": "eng", "views": {"$gte": 500}},
            include=["distance", "metadata"])

# --- filter: metadata only, every match, optionally sorted ------------------
index.query_metadata(filters={"tags": "policy"},
                     order_by="views", ascending=False, top_k=20)
```

**Operators**

| | |
|---|---|
| equality | `{"f": v}` · `$eq` · `$ne` |
| sets | `$in` · `$nin` |
| ranges (numeric) | `$gt` · `$gte` · `$lt` · `$lte` |
| presence | `$exists` |
| text (`pattern` fields) | `$regex` · `$contains` |
| logic | `$and` · `$or` · `$not` · `$nor` (+ implicit AND of top-level keys) |
| not supported | `$type`, `$size`, `$elemMatch`, `$all`, `$mod`, `$where` |

**Semantics in one line each**

- Multiple keys, and multiple operators on one field, **AND** together.
- Positive operators need the field; complements (`$nin`, `$exists: false`, `$not`, `$nor`)
  include items that lack it.
- `null`, lists of objects, and nested lists are stored but **not indexed** — they behave as absent.
- Lists index element-wise: `{"tags": "x"}` means *contains*.
- Nested dicts are dot-paths: `owner.team`.
- Ranges are numeric only; strings have no filter ordering (but `order_by` sorts them).
- `order_by`: `number < string < bool`; missing/non-scalar last in **both** directions;
  `top_k` applied **after** the sort.
- Text matching sees the **first 256 bytes** of a value.
- `query()` always answers the filter (pre-filter or post-filter). `query_metadata()` answers only
  what the index can resolve, and raises otherwise.

---
## Cleanup

In [17]:
index.delete_index()
print("index deleted")

index deleted
